## Setup — paths and environment

In [ ]:
import json

from aieng.syn_data.synbench.display import (  # noqa: E402
    show_actions,
    show_message,
    show_pipeline_messages,
    show_prompt,
    show_session,
    show_task,
)
from utils.utils import get_root_directory


# Define the root directory for this implementation
# Gets the absolute directory path of the current script
ROOT = get_root_directory()

DOMAIN_PATH = ROOT / "domains" / "mock_retail"
OUT_DIR = ROOT / "data" / "benchmarks" / "mock_retail"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Implementation root:", ROOT)
print("Domain:", DOMAIN_PATH)
print("Output:", OUT_DIR)

Implementation root: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation
Domain: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/domains/mock_retail
Output: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/data/benchmarks/mock_retail


---
## Step 1 — Load and inspect the domain

A **domain bundle** is the simulated business. `load_domain()` reads all required files and returns a `DomainBundle` object used everywhere else.

In [2]:
from aieng.syn_data.synbench.domain.loader import load_domain, validate_domain


errors = validate_domain(DOMAIN_PATH)
assert errors == [], f"Domain validation failed: {errors}"

domain = load_domain(DOMAIN_PATH)

print("Domain name:", domain.manifest.name)
print("\nTools:")
for t in domain.tools:
    print(f"  - {t.name} ({t.tool_type.value}, fsm_tag={t.fsm_tag})")
print("\nTask types (FSM):", list(domain.state_machine.get("task_types", {}).keys()))
print("\nSeed task IDs:", [t.id for t in domain.seed_tasks])
print("\nOrders in DB:", list(domain.db["orders"].keys()))

Domain name: mock_retail

Tools:
  - find_user_id (read, fsm_tag=lookup)
  - get_order (read, fsm_tag=lookup)
  - list_orders (read, fsm_tag=lookup)
  - cancel_order (write, fsm_tag=mutate)
  - update_shipping (write, fsm_tag=mutate)

Task types (FSM): ['inquiry', 'cancel', 'update_address', 'refuse_cancel']

Seed task IDs: ['seed_inquiry', 'seed_cancel', 'seed_refuse_cancel', 'seed_update_address']

Orders in DB: ['ord_1001', 'ord_1002', 'ord_2001']


In [3]:
# Peek at policy and one order
print("=== Policy (first 400 chars) ===")
print(domain.policy[:400], "...\n")

print("=== Sample order ord_1001 ===")
print(json.dumps(domain.db["orders"]["ord_1001"], indent=2))

=== Policy (first 400 chars) ===
# Mock Retail Customer Service Policy

You are a customer service agent for Mock Retail.

## Rules

1. Always look up an order before canceling or updating shipping.
2. Only cancel orders with status `pending`. Shipped or delivered orders cannot be canceled.
3. If a customer asks to cancel a non-pending order, politely refuse and explain why.
4. When updating shipping, confirm the new address with ...

=== Sample order ord_1001 ===
{
  "order_id": "ord_1001",
  "user_id": "user_alice",
  "status": "pending",
  "items": [
    "widget-a"
  ],
  "shipping_address": "123 Main St, Boston, MA"
}


---
## Step 2 - Load and re-verify previously generated tasks

In [3]:
# Re-verify all the generated tasks in the saved file.
from aieng.syn_data.synbench.schemas.tasks import Task
from aieng.syn_data.synbench.verification.pipeline import verify_draft


task_path = OUT_DIR / "tasks.json"
with open(task_path) as f:
    saved = json.load(f)

filtered_tasks = []
for item in saved["tasks"]:
    task = Task.model_validate(item)
    filtered_tasks.append(task)
    vr = verify_draft(domain, task)
    status = "OK" if vr.verification_report.passed else "FAIL"
    print(f"  [{status}] {task.id}")

# Do you see any FAILED tasks?
# If yes, import filter_verified from synbench.verification.pipeline and call it
# on (domain, filtered_tasks) to drop the failures before evaluating.

  [OK] gen_64991457_0
  [OK] gen_f0e2e807_1
  [OK] gen_3414ca87_2


---
## Step 3 — LLM layer (provider-agnostic agent core)

Agents and generation share `synbench.llm`. **`get_client()`** returns an OpenAI-compatible `ChatClient` (default model: `gemini-2.0-flash`).

Make sure to run the first notebook to check your API key works.


In [4]:
from aieng.syn_data.synbench.llm import get_client, get_model
from aieng.syn_data.synbench.llm.tools import tool_specs_to_openai


client = get_client()
print("Model:", get_model())
print("Client type:", type(client).__name__)
print("\nOpenAI-style tool schema (first tool):")

# See how the tool schema is passed to the LLM
print(json.dumps(tool_specs_to_openai(domain.tools)[3], indent=2))

Model: gemini-3.1-flash-lite-preview
Client type: ChatClient

OpenAI-style tool schema (first tool):
{
  "type": "function",
  "function": {
    "name": "cancel_order",
    "description": "Cancel a pending order",
    "parameters": {
      "type": "object",
      "properties": {
        "order_id": {
          "type": "string"
        }
      },
      "required": [
        "order_id"
      ]
    }
  }
}


---
## Step 4 — Multi-turn tool-calling loop

`ToolCallingLoop` handles **tool rounds for one user utterance**:

1. Build system prompt (policy only — no clean instructions leaked to the agent)
2. Call the LLM with domain tools
3. On tool calls → dispatch on the **live** Environment, append results
4. On text → record as an agent message and stop that utterance

User ↔ agent dialogue (multiple customer messages) is handled by `SingleToolAgent` / `AgentPipeline` via `max_dialogue_turns`.


In [5]:
# Select a task to run
selected_task = filtered_tasks[-1]
show_task(selected_task)

### Task
**id:** `gen_3414ca87_2`  
**type:** `inquiry`  
**description:** Look up an order status for the customer to confirm the state of their pending order.

#### User scenario
- **user name:** Bob Martinez
- **personality style:** domain_expert
- **instructions:** Simulator brief: The customer wants to check the status of order ord_2001. If the agent asks for your name, provide 'Bob Martinez'. Verify that the agent follows the protocol of identifying the user_id before calling get_order.
- **initial message:** I would like to verify the current status of one of my recent orders. I am aware of the company policies regarding account verification, so let's proceed through the proper channels.

#### Evaluation criteria
- **reward basis:** DB, COMMUNICATE
- **communicate info:** ['pending']
- **oracle actions:**
  - `get_order` `{"order_id": "ord_2001"}`

### [Optional] — `ToolCallingLoop` main steps

The next cell demos **one utterance** only (you can skip it).

- Tool calls → execute and feed results back so the LLM can continue.
- Agent text → end this utterance; store the reply in `agent_messages`.
- For multi-turn **customer** dialogue, use `SingleToolAgent(max_dialogue_turns=…)` in Step 5.


In [6]:
from aieng.syn_data.synbench.agents.prompts import agent_system_prompt
from aieng.syn_data.synbench.agents.session import AgentSession


task = selected_task

# ToolCallingLoop steps (one utterance demo):
# Step 0: Initialize session with task, and create a new environment
session = AgentSession(domain=domain, task=task)
env = session.ensure_env()
show_session(session, title="Session (initialized)", show_chat=False)

# Step 1: Build system prompt from policy (no instructions leak)
system_prompt = agent_system_prompt(domain, task, extra="")
show_prompt(system_prompt, title="System prompt")
session.messages.append({"role": "system", "content": system_prompt})
show_message(task.user_scenario.initial_message, role="user", title="Initial user message")
session.messages.append({"role": "user", "content": task.user_scenario.initial_message})

# Step 2: Call the LLM with domain tools (first turn call)
tools = tool_specs_to_openai(domain.tools)
response = client.complete(session.messages, tools=tools)

# Step 3: On tool calls → dispatch on live Environment, append results
if response.tool_calls:
    show_actions(response.tool_calls, title="LLM tool calls (turn 1)")
    for action in response.tool_calls:
        result = env.dispatch(action)
        result_str = json.dumps(result, default=str)
        session.agent_actions.append(action)
        session.messages.append({"role": "tool", "content": result_str})

# Step 4: A text reply ends this utterance; further customer turns are
# handled by the SingleToolAgent dialogue
if response.content:
    session.agent_messages.append(response.content)
    session.messages.append({"role": "assistant", "content": response.content})

### Session (initialized)

| Field | Value |
| --- | --- |
| **domain** | `mock_retail` (`/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/domains/mock_retail`) |
| **task** | `gen_3414ca87_2` · `inquiry` |
| **env** | ready |
| **messages** | 0 |
| **agent_actions** | 0 |
| **agent_messages** | 0 |
| **role_trace** | [] |
| **plan** | — |

### System prompt

```
You are the customer service agent for this domain.
task_id: gen_3414ca87_2

## Policy
# Mock Retail Customer Service Policy

You are a customer service agent for Mock Retail.

## Rules

1. Customers do not know their `user_id`. Always ask for the customer's full
   name first, then call `find_user_id` with that name to retrieve their
   `user_id`. Never invent, guess, or fabricate a `user_id`.
2. If `find_user_id` fails (user not found), tell the customer no account
   matches that name and ask them to confirm they typed their full name
   correctly. Do not invent a `user_id` or continue with order tools until
   a match is found.
3. Use the retrieved `user_id` with `list_orders` (and when verifying ownership)
   before answering account-level questions or modifying orders.
4. After looking up an order, confirm the order details with the customer —
   including items and shipping address — before canceling, updating shipping,
   or treating the lookup as complete.
5. Always look up an order before canceling or updating shipping.
6. Only cancel orders with status `pending`. Shipped or delivered orders cannot
   be canceled.
7. If a customer asks to cancel a non-pending order, politely refuse and explain
   why.
8. When updating shipping, confirm the new address with the customer in your
   message.
9. Never modify orders that do not belong to the requesting user.

## Communication

- Be concise and professional.
- When refusing a cancellation, state that the order cannot be canceled due to its status.


Resolve the customer's messages using the provided tools and policy.
Do not invent goals beyond what the customer says.
When finished, reply to the customer clearly.
```

### Initial user message

> I would like to verify the current status of one of my recent orders. I am aware of the company policies regarding account verification, so let's proceed through the proper channels.

---
## Step 5 — Evaluate a single agent

`SingleToolAgent` runs **user ↔ agent dialogue** (same helper as the multi-agent pipeline):

1. Turn 0 uses `user_scenario.initial_message`
2. Later turns use `UserSimulator` (`user_name` + `personality_style`) until `[[DONE]]` or `max_dialogue_turns`
3. Each utterance runs `ToolCallingLoop` (tool rounds)

Then **`score_trajectory(domain, …)`** (or `run_and_score_task`) compares:

- **db_reward** — agent DB hash vs oracle hash (actions replayed on a fresh env)
- **communicate_reward** — required substrings in agent messages
- **reward** — product of enabled `reward_basis` terms

The agent under test does **not** see `instructions` — only the live customer messages + policy.


In [7]:
from aieng.syn_data.synbench.agents.single import SingleToolAgent
from aieng.syn_data.synbench.evaluation.scoring import score_trajectory


# max_turns = tool rounds per utterance; max_dialogue_turns = customer rounds
single = SingleToolAgent(domain, max_turns=12, max_dialogue_turns=3, client=client)
session = single.run_task(selected_task)
# Prefer run_and_score_task; score_trajectory needs domain as first arg.
score = score_trajectory(domain, selected_task, session.agent_actions, session.agent_messages)

print("User name:", selected_task.user_scenario.user_name)
print("Personality style:", selected_task.user_scenario.personality_style)
print("Role trace:", session.role_trace)
print("\n=== Score breakdown ===")
print(f"  reward:              {score.reward}")
print(f"  db_reward:           {score.db_reward}")
print(f"  communicate_reward:  {score.communicate_reward}")
print(f"  partial_action_match:{score.partial_action_match:.2f}  (diagnostic only)")
print(f"  target_db_hash:      {score.target_db_hash[:24]}...")
print(f"  predicted_db_hash:   {score.predicted_db_hash[:24]}...")
if score.missing_communicate:
    print(f"  missing phrases:     {score.missing_communicate}")
    print(f"  Agent messages: {session.agent_messages}")

# Inspect the agent's messages
show_pipeline_messages(session, title="User and agent messages")

User name: Bob Martinez
Personality style: domain_expert
Role trace: ['initial_user_message', 'executor', 'user_sim', 'executor', 'user_sim', 'executor']

=== Score breakdown ===
  reward:              1.0
  db_reward:           1.0
  communicate_reward:  1.0
  partial_action_match:1.00  (diagnostic only)
  target_db_hash:      9970b161e0dd6be706f20676...
  predicted_db_hash:   9970b161e0dd6be706f20676...


### User and agent messages

**role_trace:** `initial_user_message → executor → user_sim → executor → user_sim → executor`

**[0] system**
    You are the customer service agent for this domain.
    task_id: gen_3414ca87_2

    ## Policy
    # Mock Retail Customer Service Policy

    You are a customer service agent for Mock Retail.

    ## Rules

    1. Customers do not know their `user_id`. Always ask for the customer's full
       name first, then call `find_user_id` with that name to retrieve their
       `user_id`. Never invent, guess, or fabricate a `user_id`.
    2. If `find_user_id` fails (user not found), tell the customer no account
       matches that name and ask them to confirm they typed their full name
       correctly. Do not invent a `user_id` or continue with order tools until
       a match is found.
    3. Use the retrieved `user_id` with `list_orders` (and when verifying ownership)
       before answering account-level questions or modifying orders.
    4. After looking up an order, confirm the order details with the customer —
       including items and shipping address — before canceling, updating shipping,
       or treating the lookup as complete.
    5. Always look up an order before canceling or updating shipping.
    6. Only cancel orders with status `pending`. Shipped or delivered orders cannot
       be canceled.
    7. If a customer asks to cancel a non-pending order, politely refuse and explain
       why.
    8. When updating shipping, confirm the new address with the customer in your
       message.
    9. Never modify orders that do not belong to the requesting user.

    ## Communication

    - Be concise and professional.
    - When refusing a cancellation, state that the order cannot be canceled due to its status.


    Resolve the customer's messages using the provided tools and policy.
    Do not invent goals beyond what the customer says.
    When finished, reply to the customer clearly.

**[1] user**
    I would like to verify the current status of one of my recent orders. I am aware of the company policies regarding account verification, so let's proceed through the proper channels.

**[2] assistant (executor)**
    I would be happy to help you with that. To get started, could you please provide your full name so I can look up your account?

**[3] user**
    My name is Bob Martinez. Before we move any further, I assume you need to confirm my user_id to proceed with the verification protocol? I am familiar with your internal security policies and expect the account to be properly authenticated before you attempt to access the record for order ord_2001.

**[4] assistant (executor)**
    tool_call `find_user_id`
    {
      "name": "Bob Martinez"
    }

**[5] tool**
    {
      "user_id": "user_bob",
      "name": "Bob Martinez",
      "email": "bob@example.com"
    }

**[6] assistant (executor)**
    tool_call `get_order`
    {
      "order_id": "ord_2001"
    }

**[7] tool**
    {
      "order_id": "ord_2001",
      "user_id": "user_bob",
      "status": "pending",
      "items": [
        "gadget-x"
      ],
      "shipping_address": "789 Pine Rd, Austin, TX"
    }

**[8] assistant (executor)**
    Thank you, Bob. I have authenticated your account and retrieved order ord_2001.

    The order contains:
    - Item: gadget-x
    - Shipping Address: 789 Pine Rd, Austin, TX
    - Current Status: pending

    Is this the information you were looking for, or did you need assistance with something else regarding this order?

**[9] user**
    That is the correct order. However, I am curious why the status is still listed as "pending." My records indicate that the fulfillment window for a gadget-x should have closed 48 hours ago. Could you please clarify if there is a specific bottleneck in the warehouse or a pending payment authorization issue causing this delay?

**[10] assistant (executor)**
    I apologize for the confusion regarding the status of your order. While I don't have access to the specific internal details regarding warehouse operations or payment processing bottlenecks, I can confirm that the order is currently marked as "pending" in our system.

    If you are concerned about a payment authorization issue, I recommend checking the payment method on file. Otherwise, I can certainly monitor this for you. Would you like me to check back on this for you, or is there anything else I can assist you with regarding this order today?


In [8]:
# Score seed tasks (run_and_score_task → ScoreResult, not AgentSession)
for task in domain.seed_tasks:
    s = single.run_and_score_task(task)
    style = task.user_scenario.personality_style or "—"
    print(f"{task.id:20s} style={style:14s} reward={s.reward}  db={s.db_reward}  comm={s.communicate_reward}")

seed_inquiry         style=rushed         reward=1.0  db=1.0  comm=1.0
seed_cancel          style=anxious        reward=1.0  db=1.0  comm=1.0
seed_refuse_cancel   style=rule_breaker   reward=0.0  db=1.0  comm=0.0
seed_update_address  style=domain_expert  reward=1.0  db=1.0  comm=1.0


---
## Step 6 — Batch metrics (pass@1)

`MetricsCollector` aggregates scores across task

In [9]:
from aieng.syn_data.synbench.evaluation.metrics import MetricsCollector


# Evaluate generated tasks with single agent
metrics = MetricsCollector()
for task in filtered_tasks:
    metrics.add(task.id, single.run_and_score_task(task))

summary = metrics.summary()
print(json.dumps(summary, indent=2))

{
  "n_tasks": 3,
  "pass_at_1": 0.6666666666666666,
  "mean_db_reward": 1.0,
  "mean_communicate_reward": 0.6666666666666666,
  "mean_partial_action_match": 1.0,
  "runs": [
    {
      "task_id": "gen_64991457_0",
      "reward": 1.0,
      "db_reward": 1.0,
      "communicate_reward": 1.0,
      "partial_action_match": 1.0
    },
    {
      "task_id": "gen_f0e2e807_1",
      "reward": 0.0,
      "db_reward": 1.0,
      "communicate_reward": 0.0,
      "partial_action_match": 1.0
    },
    {
      "task_id": "gen_3414ca87_2",
      "reward": 1.0,
      "db_reward": 1.0,
      "communicate_reward": 1.0,
      "partial_action_match": 1.0
    }
  ]
}
